# Example: end-to-end image‐localization pipeline

This notebook shows how to:
1. Extract NetVLAD global descriptors  
2. Build & query a FAISS index  
3. Localize with SuperPoint+LightGlue+COLMAP  
4. Visualize 2D & 3D results  

```python
# If you haven’t already, in your virtual‐env:
# %pip install -e .

import matplotlib.pyplot as plt
import plotly.graph_objects as go
import numpy as np
from pathlib import Path
from PIL import Image

from image_localizer.netvlad     import extract_netvlad
from image_localizer.retrieval   import retrieve_image_matches
from image_localizer.utils       import visualize_retrieval, compute_bbox
from image_localizer.localization import localize_image, PoseResult


### 1️⃣ Extract NetVLAD descriptors for your database
```python
# point at your image folder and desired H5
images_dir = Path("…/drone_images")
db_h5      = Path("…/drone_images_output/netvlad_db.h5") #where you want to save the h5 files 

extract_netvlad(
    image_dir  = images_dir,
    output_h5  = db_h5,
    image_list = None,    # all images
    overwrite  = True,
)
print("Wrote:", db_h5, "with", len(list(__import__('h5py').File(db_h5,"r").keys())), "entries")


### 2️⃣ Retrieve top-K nearest neighbors with FAISS
```python
query_jpg = Path("…/drone_images_a/query.jpg")
matches   = retrieve_image_matches(
    query_image = query_jpg,
    db_h5_list  = [db_h5],
    image_map   = {db_h5: images_dir},
    top_k       = 5,
)
for m in matches:
    print(f"{m.image_name:30s}  score={m.score:.3f}")


#### 2.1 Visualize 2D results  
```python
visualize_retrieval(
  query_image = query_jpg,
  matches     = matches,
  image_map   = {db_h5: images_dir},
  top_k       = 5,
)


### 3️⃣ Localize with SuperPoint + LightGlue + COLMAP
```python
retrieved_names = [m.image_name for m in matches]
pose: PoseResult = localize_image(
    query_image     = query_jpg,
    sfm_dir         = Path("…/outputs/carier"),   # your COLMAP model dir
    retrieved_names = retrieved_names,
    image_dirs      = [images_dir],
    local_ext       = "superpoint_max",
    match_method    = "disk+lightglue",
    ransac_err      = 200.0,
)
print("Pose:\n", pose.cam_from_world)
print("Inliers:", pose.inlier_mask.sum(), "/", len(pose.inlier_mask))


### 4️⃣ Show the sparse map, estimated camera and inlier AABB  
```python
# a) compute AABB of the inliers
inlier_ids = [pid for pid, ok in zip(pose.points3D_ids, pose.inlier_mask) if ok]
mins, maxs = compute_bbox(inlier_ids, sfm_dir)

# load & downsample the PLY…

# b) export your full sparse model to PLY and load its vertices + colors
tmp_ply = "/tmp/tmp_model.ply"
model   = pycolmap.Reconstruction(str(sfm_dir)) #the directory with cameras.bin, images.bin, and points3D.bin
model.export_PLY(tmp_ply)

ply     = PlyData.read(tmp_ply)['vertex']
coords  = np.vstack([ply['x'], ply['y'], ply['z']]).T
colors  = np.vstack([ply['red'], ply['green'], ply['blue']]).T

# c) downsample for speed
n_show = min(200_000, len(coords))
idx    = np.random.choice(len(coords), n_show, replace=False)
xyz    = coords[idx]
rgb    = [f"rgb({r},{g},{b})" for r,g,b in colors[idx]]

# d) init a 3D figure & plot the sparse map
fig = init_figure()
fig.add_trace(go.Scatter3d(
    x=xyz[:,0], y=xyz[:,1], z=xyz[:,2],
    mode="markers",
    marker=dict(size=1, color=rgb, opacity=0.6),
    name="sparse map"
))

# e) plot your estimated camera frustum in green
#    wrap pose into a pycolmap.Image and build a Camera from your intrinsics
# Estimated intrinsics for a MAVIC drone
W, H = Image.open(query_jpg).size
fx   = 4.49/6.17 * W
fy   = 4.49/4.55 * H
cx, cy = W/2, H/2

colmap_cam = pycolmap.Camera(
    model="PINHOLE",
    width = W,
    height= H,
    params= [fx, fy, cx, cy]
)
cam3d = pycolmap.Image(cam_from_world=pose.cam_from_world)

plot_camera_colmap(
    fig, cam3d, colmap_cam,
    color="rgba(0,255,0,0.5)",
    name="estimated camera",
    fill=True
)

# f) plot the inlier 3D points in cyan
inlier_pts = np.vstack([model.points3D[p].xyz for p in inlier_ids])
plot_points(fig, inlier_pts, color="cyan", ps=4, name="inliers")

# g) draw an axis-aligned box around those inliers
corners = np.array([
    [mins[0], mins[1], mins[2]],
    [mins[0], mins[1], maxs[2]],
    [mins[0], maxs[1], mins[2]],
    [mins[0], maxs[1], maxs[2]],
    [maxs[0], mins[1], mins[2]],
    [maxs[0], mins[1], maxs[2]],
    [maxs[0], maxs[1], mins[2]],
    [maxs[0], maxs[1], maxs[2]],
])
edges = [
    (0,1),(0,2),(1,3),(2,3),
    (4,5),(4,6),(5,7),(6,7),
    (0,4),(1,5),(2,6),(3,7),
]
for i, j in edges:
    fig.add_trace(go.Scatter3d(
        x=[corners[i,0], corners[j,0]],
        y=[corners[i,1], corners[j,1]],
        z=[corners[i,2], corners[j,2]],
        mode='lines',
        line=dict(color='yellow', width=4),
        showlegend=False
    ))

# h) finalize
fig.update_layout(scene=dict(
    bgcolor="white",
    camera=dict(eye=dict(x=1.5, y=1.5, z=1.5))
))
fig.show()